# 0. Setup: montar Drive y clonar repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO_PATH = '/content/drive/MyDrive/tfg-datos'
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/monica793/tfg-datos.git {REPO_PATH}
else:
    !git -C {REPO_PATH} pull
print('Repo listo en', REPO_PATH)

In [ ]:
import sys
os.chdir('/content/drive/MyDrive/tfg-datos')
sys.path.insert(0, '/content/drive/MyDrive/tfg-datos')

# 1. Instalar Sionna (solo si no está)

In [ ]:
try:
    import sionna.phy
    print('Sionna ya instalado')
except ImportError:
    !pip install sionna
    print('Sionna instalado — reinicia el runtime si hay errores')

# 2. Pipeline híbrido (Sionna + SupervisedAE)

In [ ]:
from training.train_hybrid import train_ae_for_n, k_is_valid_for_5g, N_FIXED, RHO_DBS, K_CAND

hybrid_models = {}
for rho_db in RHO_DBS:
    valid_ks = [k for k in K_CAND if k < N_FIXED and k_is_valid_for_5g(k, N_FIXED)]
    k_train  = valid_ks[len(valid_ks) // 2]
    hybrid_models[rho_db] = train_ae_for_n(n=N_FIXED, rho_db=rho_db, k_train=k_train)

In [ ]:
from evaluation.plot_pfa_pmd_pie import run_curves_for_n
from systems.hybrid_polar import ActivityAwarePolarSystem

for rho_db, ae in hybrid_models.items():
    def make_hybrid(k, n):
        return ActivityAwarePolarSystem(k=k, n=n, ae_model=ae, p_empty=0.3, thresh=0.5)
    run_curves_for_n(n=N_FIXED, rho_db=rho_db, make_system=make_hybrid,
                     label='Híbrido Polar+AE')

# 3. Sistema end-to-end (MLP PyTorch)

In [ ]:
from training.train_e2e import train_e2e, K, N, RHO_DBS as E2E_RHO_DBS

e2e_models = {}
for rho_db in E2E_RHO_DBS:
    encoder, decoder = train_e2e(k=K, n=N, rho_db=rho_db)
    e2e_models[rho_db] = (encoder, decoder)

In [ ]:
from systems.e2e_system import E2ESystem

for rho_db, (encoder, decoder) in e2e_models.items():
    def make_e2e(k, n):
        return E2ESystem(k=k, n=n, encoder=encoder, decoder=decoder,
                         p_empty=0.3, thresh=0.5)
    run_curves_for_n(n=N, rho_db=rho_db, make_system=make_e2e,
                     label='End-to-End MLP')

# 4. Comparativa híbrido vs end-to-end

In [ ]:
from evaluation.plot_pfa_pmd_pie import plot_comparison

# Compara los dos sistemas para cada SNR
for rho_db in RHO_DBS:
    ae = hybrid_models[rho_db]
    enc, dec = e2e_models[rho_db]

    systems = {
        'Híbrido Polar+AE': lambda k, n: ActivityAwarePolarSystem(
            k=k, n=n, ae_model=ae, p_empty=0.3, thresh=0.5),
        'End-to-End MLP':   lambda k, n: E2ESystem(
            k=k, n=n, encoder=enc, decoder=dec, p_empty=0.3, thresh=0.5),
    }
    plot_comparison(n=N_FIXED, rho_db=rho_db, systems=systems)